# Cortex-Local 蒸馏训练（Colab G4 / A100 通用版）

在 Colab GPU 上跑完 `Eveyz/cortex-rs` 全流程：先用 Qwen3 老师生成经过验证的蒸馏数据（`distill_generate.py`），再训练 1.58-bit 学生（`distill_train.py`），最后把 `.ctxb` 存到 Google Drive。

## GPU 选型与推荐配置（当前默认针对 **Colab G4 96GB** 优化）

| Colab GPU | 显存 | 老师模型 | 学生训练（`production` = 1.6B） |
|---|---|---|---|
| **G4（RTX PRO 6000 Blackwell，推荐）** | **96GB GDDR7** | 默认 `Qwen/Qwen3-30B-A3B-Instruct-2507`（bf16 约 60GB，余量 36GB，极快） | 默认 `--config production --batch-size 2 --grad-accum 4`（约 55GB，稳；可提速到 `--batch-size 4 --grad-accum 2`） |
| A100 80GB（Pro+） | 80GB | 默认 `Qwen/Qwen3-30B-A3B-Instruct-2507`（bf16 约 60GB） | 默认 `--config production --batch-size 2 --grad-accum 4` |
| A100 40GB / L4 | 40 / 24GB | `--teacher Qwen/Qwen3-30B-A3B-AWQ` 或 `Qwen/Qwen3-8B` | 24GB 需降参：`--batch-size 1 --grad-accum 8 --seq-len 256` |
| T4（免费） | 16GB | `--teacher Qwen/Qwen3-4B` | 仅建议 `--config micro` 冒烟 |

## 跑前必看

- **Google Drive 空间**：`production` 训学生时 `latest.pt` 约 **19GB**（原子覆盖写瞬间峰值约 **38GB**，已默认关闭 `step-*.pt` 堆叠防爆盘），请确保 Drive 至少有 **40GB 可用空间**。
- **全程断点续跑**：数据生成按 `prompt + response` 双重去重跳过已做题目；训练每 30 分钟原子刷新 `latest.pt`，Colab 到点断开后重连直接重跑对应 Cell 即可接着走。

In [ ]:
# Cell 1: 检查 GPU 型号（应看到 RTX PRO 6000 / ~96GB）与本地磁盘空间（建议 >= 80GB 放 60GB 老师缓存）
!nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv
!df -h /content | tail -1

In [ ]:
# Cell 2: 挂载 Google Drive（所有数据、checkpoint、最终 student.ctxb 均持久化在这里）
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/cortex-local'
os.makedirs(WORK, exist_ok=True)
print('work dir:', WORK)
# 检查 Drive 剩余空间（训 1.6B production 建议 >= 40GB）
!df -h /content/drive | tail -1

In [ ]:
# Cell 3: 拉取最新 main 分支代码
%cd /content
!test -d cortex-rs || git clone https://github.com/Eveyz/cortex-rs.git
%cd /content/cortex-rs
!git pull --ff-only origin main
!git log --oneline -5

In [ ]:
# Cell 4: 安装依赖：vLLM + hf_transfer（高速下载 60GB 老师权重）+ datasets（GSM8K/MBPP 真题库）
# 注意：如果 Colab 提示「重新启动会话 / Restart session」，点击重启后重新执行 Cell 2 和 Cell 3，再直接跳到 Cell 5
%pip install -q -U vllm hf_transfer datasets

import torch, vllm
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), 'compute_cap:', torch.cuda.get_device_capability(0))
    # 快速验证 Blackwell (sm_120) CUDA kernel 可正常执行
    x = torch.randn(64, 64, device='cuda', dtype=torch.bfloat16)
    print('cuda bf16 matmul check ok, norm =', float((x @ x).norm().item()))
print('vllm:', vllm.__version__)

In [ ]:
# Cell 5: 冒烟测试（每类最多 20 条，验证 G4 跑 Qwen3-30B-A3B 速度与数据格式）
# --math-source gsm8k --code-source mbpp：全用 train 集真题，零 HumanEval 评测集污染
# （如不关心 HumanEval 基准污染，也可改成 --code-source humaneval+mbpp）
import os, json
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HOME'] = '/content/.cache/huggingface'

TEACHER = 'Qwen/Qwen3-30B-A3B-Instruct-2507'
WORK = '/content/drive/MyDrive/cortex-local'
%cd /content/cortex-rs

!python tools/distill_generate.py \
  --teacher {TEACHER} \
  --out {WORK}/distill-data.jsonl \
  --limit 20 \
  --math-source gsm8k \
  --code-source mbpp

# 抽查前 2 条样本格式
with open(f'{WORK}/distill-data.jsonl', encoding='utf-8') as f:
    for i in range(2):
        line = f.readline()
        if not line: break
        rec = json.loads(line)
        print(f'--- sample {i} [{rec["category"]}] verified={rec["verified"]} ---')
        print('prompt:', rec['prompt'][:160])
        print('response:', rec['response'][:200])

In [ ]:
# Cell 6: 全量蒸馏数据生成（约 1.45 万条带验证数据，后台 nohup 运行）
# 断线重连后直接重跑本 Cell，会自动跳过已有 prompt+response，绝不重复生成
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HOME'] = '/content/.cache/huggingface'

TEACHER = 'Qwen/Qwen3-30B-A3B-Instruct-2507'
WORK = '/content/drive/MyDrive/cortex-local'
%cd /content/cortex-rs

!nohup python tools/distill_generate.py \
  --teacher {TEACHER} \
  --out {WORK}/distill-data.jsonl \
  --math-source gsm8k \
  --code-source mbpp \
  > {WORK}/gen.log 2>&1 &
print('started in background, log:', f'{WORK}/gen.log')

In [ ]:
# Cell 7: 随时查看数据生成进度、分类计数和 GPU 占用
import json
from collections import Counter
WORK = '/content/drive/MyDrive/cortex-local'

!ps aux | grep '[d]istill_generate.py' || echo 'distill_generate 已结束（或未启动）'
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv
!tail -10 {WORK}/gen.log 2>/dev/null || echo 'no gen.log yet'

p = f'{WORK}/distill-data.jsonl'
try:
    c = Counter(json.loads(ln).get('category', '?') for ln in open(p, encoding='utf-8') if ln.strip())
    print('total lines:', sum(c.values()), 'by category:', dict(c))
except FileNotFoundError:
    print('no distill-data.jsonl yet')

In [ ]:
# Cell 8: 训练 1.58-bit 学生模型（production = 1.6B，2 epochs，后台 nohup 运行）
# 跑本 Cell 前，先确认 Cell 7 里 distill_generate 已结束且 GPU 显存已释放！
# G4 96GB 默认 --batch-size 2 --grad-accum 4（约占 55GB 显存，非常稳）；
# 如果 Cell 9 里看到显存充足，下一轮可改成 --batch-size 4 --grad-accum 2 提速约 1.8x。
# 每 30 分钟原子覆盖保存 Drive 上的 latest.pt（约 19GB）；断线重连重跑本 Cell 自动续训！
import os, subprocess
WORK = '/content/drive/MyDrive/cortex-local'
CKPT = f'{WORK}/ckpts'
%cd /content/cortex-rs

# 防呆检查：确认没有残留的 distill_generate 占着显存
ps_gen = subprocess.run("ps aux | grep '[d]istill_generate.py'", shell=True, capture_output=True, text=True)
assert not ps_gen.stdout.strip(), 'distill_generate.py 仍在运行中！请等它跑完释放显存后再启动训练。'

resume = f'--init-from {CKPT}/latest.pt' if os.path.exists(f'{CKPT}/latest.pt') else ''
print('resume:', resume or '(fresh start)')

!nohup python tools/distill_train.py \
  --data {WORK}/distill-data.jsonl \
  --config production \
  --epochs 2 \
  --batch-size 2 \
  --grad-accum 4 \
  --seq-len 512 \
  --out {WORK}/student.ctxb \
  --ckpt-dir {CKPT} \
  --ckpt-every-min 30 \
  {resume} \
  > {WORK}/train.log 2>&1 &
print('started in background, log:', f'{WORK}/train.log')

In [ ]:
# Cell 9: 随时查看学生训练进度、GPU 显存与 checkpoint
WORK = '/content/drive/MyDrive/cortex-local'
!ps aux | grep '[d]istill_train.py' || echo 'distill_train 已结束（或未启动）'
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv
!tail -15 {WORK}/train.log 2>/dev/null || echo 'no train.log yet'
!ls -lh {WORK}/ckpts/latest.pt {WORK}/student.ctxb 2>/dev/null

## 跨天续跑 & 拿回本地运行

### Colab 会话断开后怎么续跑
Colab Pro/Pro+ 单次运行时最长约 24 小时，断开后只需：
1. 重新选择 **G4 GPU** 运行时；
2. 依次运行 **Cell 2**（挂 Drive）→ **Cell 3**（`git pull`）→ **Cell 4**（装依赖，仅数据生成阶段需要 `vllm`；如果已在训练阶段，其实只需 `datasets` 或直接跳到 Cell 8，因为 Colab 自带 `torch`）；
3. 如果数据还没生完，重跑 **Cell 6**；如果已经在训学生，直接重跑 **Cell 8**（会自动检测 `ckpts/latest.pt` 接着恢复模型、优化器、step 与 LR 曲线）。

### 训完拿回本地 Mac 运行
训练完成后，从 Google Drive 下载 `cortex-local/student.ctxb` 到本地，用纯 Rust 无依赖运行时直接开聊：
```bash
./target/release/cortex-cli repl --model /path/to/student.ctxb
# 或启用工具 Agent：
./target/release/cortex-cli agent --model /path/to/student.ctxb --prompt "帮我算 12*3+5"
```